# MIST-HER2: guided minibatch OT-CFM

Generate HER2 from Gaussian noise with H&E conditioning. This exploratory run uses a multiscale H&E encoder and TorchCFM minibatch OT. Its saved metrics cover 12 validation crops, not the full test set.


In [ ]:
SESSION_STEPS = 10_000       # 0: только inference с уже обученным V3 checkpoint
RESUME_FROM = None         # /kaggle/input/.../latest.pt; первый запуск — None
MICRO_BATCH = 2            # 1 при нехватке VRAM; OT-batch всё равно 8
SMOKE_TEST = False

In [ ]:
import importlib.metadata
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import zipfile

INPUT_ROOT = Path(os.environ.get("IHC_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("IHC_WORK_ROOT", "/kaggle/working"))
WORK_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(WORK_ROOT / ".matplotlib"))
# POT's exact EMD uses CPU; don't initialize unused ML backends on the GPU.
for backend in ("JAX", "TENSORFLOW", "CUPY"):
    os.environ[f"POT_BACKEND_DISABLE_{backend}"] = "1"

markers = list(INPUT_ROOT.rglob("kaggle_bundle.json"))
if len(markers) != 1:
    raise FileNotFoundError("Подключите один распакованный MIST dataset с kaggle_bundle.json")
BUNDLE = markers[0].parent
sys.path.insert(0, str(BUNDLE / "code"))
DATA_ROOT = BUNDLE / "data" / "HER2"
MANIFEST = BUNDLE / "manifests" / "group-disjoint" / "pairs.jsonl"

required_versions = {"torchcfm": "1.0.7", "POT": "0.9.6.post1"}
def installed(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
if any(installed(name) != version for name, version in required_versions.items()):
    wheel_markers = list(INPUT_ROOT.rglob("torchcfm_wheels.json"))
    if not wheel_markers:
        archives = list(INPUT_ROOT.rglob("torchcfm-kaggle-wheels.zip"))
        if len(archives) == 1:
            extracted = Path(tempfile.gettempdir()) / "ihc-otcfm-wheels"
            extracted.mkdir(exist_ok=True)
            with zipfile.ZipFile(archives[0]) as archive:
                for info in archive.infolist():
                    if not (extracted / info.filename).resolve().is_relative_to(extracted.resolve()):
                        raise ValueError("Unsafe dependency ZIP path")
                archive.extractall(extracted)
            wheel_markers = list(extracted.rglob("torchcfm_wheels.json"))
    command = [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
               "torchcfm==1.0.7", "POT==0.9.6.post1"]
    if wheel_markers:
        command += ["--no-index", "--find-links", str(wheel_markers[0].parent)]
    subprocess.run(command, check=True)
print("torchCFM:", installed("torchcfm"), "POT:", installed("POT"))

In [ ]:
"""Image-guided minibatch OT-CFM using torchCFM 1.0.7.

The Kaggle notebook displays this code directly. The MIST bundle provides only
paired data transforms and small, already-tested model/runtime building blocks.
"""

from __future__ import annotations

import copy
import json
import math
import random
from itertools import pairwise
from pathlib import Path

import numpy as np
import torch
from ihc_flow.data import PairedDataset, TrainingBatches
from ihc_flow.download import sha256_file
from ihc_flow.model import BlockStack, TimeEmbedding, group_norm
from ihc_flow.runtime import initial_noise, save_image, seed_everything, write_json
from skimage.metrics import structural_similarity
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchcfm.conditional_flow_matching import (
    ExactOptimalTransportConditionalFlowMatcher,
)
from tqdm.auto import tqdm


def recipe(smoke: bool = False) -> dict:
    """Fixed experiment defaults, recorded in every checkpoint."""
    return {
        "method": "guided_minibatch_otcfm_v3",
        "source": "standard_gaussian",
        "sigma": 0.0,
        "torchcfm": "1.0.7",
        "pot": "0.9.6.post1",
        "crop": 64 if smoke else 512,
        "widths": (8, 16) if smoke else (32, 64, 128, 256),
        "ot_batch": 4 if smoke else 8,
        "total_steps": 8 if smoke else 20000,
        "lr": 0.001 if smoke else 0.0001,
        "warmup": 0 if smoke else 500,
        "ema": 0.9 if smoke else 0.999,
        "validate_every": 4 if smoke else 500,
        "validation_count": 4 if smoke else 12,
        "ode_steps": 2 if smoke else 32,
        "limit_train": 32 if smoke else None,
        "seed": 42,
    }


def rgb(image):
    return (image.detach().float().cpu().clamp(-1, 1).permute(1, 2, 0).numpy() + 1) / 2

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
from PIL import Image

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda" and not SMOKE_TEST:
    raise RuntimeError("Включите GPU либо выберите SMOKE_TEST=True")
torch.set_num_threads(4)
SETTINGS = recipe(SMOKE_TEST)
RUN = WORK_ROOT / ("her2-otcfm-v3-smoke" if SMOKE_TEST else "her2-otcfm-v3")
print("Device:", DEVICE, "Output:", RUN)
print("Training crop:", SETTINGS["crop"], "OT batch:", SETTINGS["ot_batch"])
with (MANIFEST.parent / "audit.json").open() as stream:
    audit = json.load(stream)
print("Split counts:", audit["counts"])
assert not any(audit["group_overlap"].values())

## Multiscale H&E conditioning

A separate encoder supplies H&E features at each U-Net scale. The network predicts a three-channel velocity field.


In [ ]:
class HEEncoder(nn.Module):
    """A spatial feature pyramid that never receives x_t or time."""

    def __init__(self, widths):
        super().__init__()
        self.input = nn.Conv2d(3, widths[0], 3, padding=1)
        self.blocks = nn.ModuleList(
            [
                nn.Sequential(
                    group_norm(c),
                    nn.SiLU(),
                    nn.Conv2d(c, c, 3, padding=1),
                    group_norm(c),
                    nn.SiLU(),
                    nn.Conv2d(c, c, 3, padding=1),
                )
                for c in widths
            ]
        )
        self.down = nn.ModuleList(
            [nn.Conv2d(a, b, 3, stride=2, padding=1) for a, b in pairwise(widths)]
        )

    def forward(self, he):
        hidden, features = self.input(he), []
        for level, block in enumerate(self.blocks):
            hidden = hidden + block(hidden)
            features.append(hidden)
            if level < len(self.down):
                hidden = self.down[level](hidden)
        return features


class UNet(nn.Module):
    """Predict RGB velocity from x_t, t and multiscale H&E features."""

    def __init__(self, widths=(32, 64, 128, 256)):
        super().__init__()
        time_dim = widths[0] * 4
        self.he_encoder = HEEncoder(widths)
        self.time = TimeEmbedding(time_dim)
        self.input = nn.Conv2d(6, widths[0], 3, padding=1)
        self.fusion = nn.ModuleList([nn.Conv2d(2 * c, c, 1) for c in widths])
        self.encoder = nn.ModuleList([BlockStack(c, c, 2, time_dim, 0) for c in widths])
        self.down = nn.ModuleList(
            [nn.Conv2d(a, b, 3, stride=2, padding=1) for a, b in pairwise(widths)]
        )
        self.middle = BlockStack(widths[-1], widths[-1], 2, time_dim, 0)
        reverse = tuple(reversed(widths))
        self.decoder = nn.ModuleList(
            [
                BlockStack((reverse[i - 1] if i else c) + 2 * c, c, 2, time_dim, 0)
                for i, c in enumerate(reverse)
            ]
        )
        self.output = nn.Sequential(
            group_norm(widths[0]), nn.SiLU(), nn.Conv2d(widths[0], 3, 3, padding=1)
        )
        nn.init.zeros_(self.output[-1].weight)
        nn.init.zeros_(self.output[-1].bias)

    def forward(self, image, time, he, features=None):
        if image.shape != he.shape or image.ndim != 4 or image.shape[1] != 3:
            raise ValueError("x_t and H&E must have matching [B,3,H,W] shapes")
        features = self.he_encoder(he) if features is None else features
        embedding = self.time(time.reshape(-1).expand(len(image)))
        hidden, skips = self.input(torch.cat((image, he), 1)), []
        for level, (fusion, block) in enumerate(zip(self.fusion, self.encoder, strict=True)):
            hidden = block(fusion(torch.cat((hidden, features[level]), 1)), embedding)
            skips.append(hidden)
            if level < len(self.down):
                hidden = self.down[level](hidden)
        hidden = self.middle(hidden, embedding)
        for block, skip, feature in zip(
            self.decoder, reversed(skips), reversed(features), strict=True
        ):
            hidden = F.interpolate(hidden, size=skip.shape[-2:], mode="nearest")
            hidden = block(torch.cat((hidden, skip, feature), 1), embedding)
        return self.output(hidden)

## Minibatch optimal transport

Match Gaussian sources to IHC targets within each minibatch. Move the H&E condition with its target to preserve the real pair. OT is used during training only; inference starts from noise with fixed H&E guidance.


In [ ]:
def ot_batch(matcher, he, ihc):
    """Use the library's guided API; H&E must follow the selected IHC endpoint."""
    source = torch.randn_like(ihc)
    time, state, velocity, _, matched_he = matcher.guided_sample_location_and_conditional_flow(
        source, ihc, y1=he
    )
    return time, state, velocity, matched_he


@torch.inference_mode()
def generate(model, he, noise, steps=32, trajectory=False):
    """Deterministic Heun ODE after the one initial Gaussian draw. No OT at inference."""
    if steps < 1 or model.training:
        raise ValueError("Use model.eval() and a positive ODE step count")
    if he.shape != noise.shape:
        raise ValueError("Initial noise and H&E shapes differ")
    image, delta = noise.clone(), 1 / steps
    features = model.he_encoder(he)
    frames = [image.cpu().clone()] if trajectory else []
    for step in range(steps):
        time = torch.full((len(image),), step / steps, device=image.device)
        first = model(image, time, he, features)
        second = model(image + delta * first, time + delta, he, features)
        image = image + delta * (first + second) / 2
        if trajectory and (step + 1) in {max(1, round(steps * f)) for f in (0.25, 0.5, 0.75, 1)}:
            frames.append(image.cpu().clone())
    if not torch.isfinite(image).all():
        raise FloatingPointError("Non-finite ODE endpoint")
    return (image, frames) if trajectory else image

## Transport plan and training path

Inspect the RGB cost matrix, transport plan and selected indices. The reference interpolation illustrates training, not inference.


In [ ]:
seed_everything(42)
train_preview = PairedDataset(DATA_ROOT, MANIFEST, "train", SETTINGS["crop"])
indices = np.linspace(0, len(train_preview) - 1, SETTINGS["ot_batch"], dtype=int)
items = [train_preview[int(index)] for index in indices]
he = torch.stack([item["he"] for item in items])
ihc = torch.stack([item["ihc"] for item in items])
z = torch.randn_like(ihc)
matcher = ExactOptimalTransportConditionalFlowMatcher(sigma=0.0)
plan = matcher.ot_sampler.get_map(z, ihc)
cost = torch.cdist(z.flatten(1), ihc.flatten(1)).square().numpy()
METHOD_DIR = WORK_ROOT / "otcfm-method"
METHOD_DIR.mkdir(exist_ok=True)
labels = torch.arange(len(z))
t, xt, ut, source_ids, target_ids = matcher.guided_sample_location_and_conditional_flow(
    z, ihc, y0=labels, y1=labels, t=torch.zeros(len(z)))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, array, title in zip(axes, (cost, plan), ("Squared RGB L2 cost", "Exact minibatch OT plan")):
    im = ax.imshow(array); ax.set_title(title)
    ax.set(xlabel="IHC j", ylabel="Noise i"); fig.colorbar(im, ax=ax)
fig.tight_layout(); fig.savefig(METHOD_DIR/"ot-plan.png",dpi=140); plt.show(); plt.close(fig)
display([{"noise_index": int(i), "target_index": int(j), "H&E/IHC id": items[int(j)]["id"]}
         for i,j in zip(source_ids,target_ids)])
write_json(METHOD_DIR/"matched-pairs.json",[
    {"noise_index":int(i),"target_index":int(j),"paired_id":items[int(j)]["id"]}
    for i,j in zip(source_ids,target_ids)])
i,j = int(source_ids[0]),int(target_ids[0])
fig, axes = plt.subplots(1, 7, figsize=(18, 3))
panels = [(he[j], "Paired H&E")] + [((1-time)*z[i]+time*ihc[j], f"Teacher t={time:g}")
                                    for time in (0,0.25,0.5,0.75,1)] + [(ihc[j], "Real IHC")]
for ax,(image,title) in zip(axes,panels):
    ax.imshow(rgb(image)); ax.set_title(title); ax.axis("off")
fig.tight_layout(); fig.savefig(METHOD_DIR/"teacher-path.png",dpi=140); plt.show(); plt.close(fig)
del items, he, ihc, z, xt, ut

## Training and checkpoints

Use native 512-pixel crops, OT batch 8, AdamW, EMA and Heun32. Micro-batches split backpropagation without changing the OT plan. The best checkpoint is selected by validation SSIM.


In [ ]:
def validation_data(root, manifest, settings):
    """Fixed group-balanced validation selection; no test data or quality-based selection."""
    dataset = PairedDataset(root, manifest, "val", settings["crop"])
    rng, grouped = random.Random(settings["seed"]), {}
    for record in dataset.records:
        grouped.setdefault(record["group"], []).append(record)
    groups = sorted(grouped)
    rng.shuffle(groups)
    for records in grouped.values():
        rng.shuffle(records)
    selected = []
    while len(selected) < min(settings["validation_count"], len(dataset)):
        for group in groups:
            if grouped[group] and len(selected) < settings["validation_count"]:
                selected.append(grouped[group].pop())
    dataset.records = selected
    return dataset


def pixel_scores(prediction, target):
    a, b = rgb(prediction), rgb(target)
    mse = float(np.square(a - b).mean())
    return {
        "rgb_mse": mse,
        "ssim": float(structural_similarity(a, b, channel_axis=-1, data_range=1)),
        "psnr": float(-10 * np.log10(mse)) if mse > 0 else None,
    }


@torch.inference_mode()
def validate(model, dataset, device, settings, directory):
    model.eval()
    rows = []
    for index in range(len(dataset)):
        item = dataset[index]
        he = item["he"].unsqueeze(0).to(device)
        noise = initial_noise(he, [item["id"]], settings["seed"])
        prediction = generate(model, he, noise, settings["ode_steps"])[0].cpu()
        rows.append({"id": item["id"], **pixel_scores(prediction, item["ihc"])})
        if index < 4:
            save_image(
                torch.cat((item["he"], prediction, item["ihc"]), 2),
                directory / f"example-{index}.png",
            )
    write_json(directory / "metrics.json", rows)
    return {
        "val_ssim": float(np.mean([r["ssim"] for r in rows])),
        "val_rgb_mse": float(np.mean([r["rgb_mse"] for r in rows])),
    }


@torch.no_grad()
def optimizer_update(model, optimizer, scaler):
    """Let GradScaler skip an overflow before advancing EMA or the step counter."""
    scaler.unscale_(optimizer)
    parameters = [p for p in model.parameters() if p.grad is not None]
    finite = torch.stack([torch.isfinite(p.grad).all() for p in parameters]).all().item()
    if not finite:
        if not scaler.is_enabled():
            raise FloatingPointError("Non-finite gradients with AMP disabled")
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        return False
    # Finite gradients can still overflow a FP32 norm reduction.
    norm = torch.linalg.vector_norm(
        torch.stack([torch.linalg.vector_norm(p.grad.to(torch.float64)) for p in parameters])
    )
    coefficient = (1 / (norm + 1e-6)).clamp(max=1)
    for parameter in parameters:
        parameter.grad.mul_(coefficient.to(parameter.grad))
    scaler.step(optimizer)
    scaler.update()
    return True


def numpy_rng_state():
    name, keys, position, has_gauss, cached = np.random.get_state()
    return name, keys.tolist(), position, has_gauss, cached


def restore_numpy_rng(state):
    name, keys, position, has_gauss, cached = state
    np.random.set_state((name, np.asarray(keys, dtype=np.uint32), position, has_gauss, cached))


def save_checkpoint(path, payload):
    temporary = path.with_suffix(".tmp")
    torch.save(payload, temporary)
    temporary.replace(path)


def train_session(
    root,
    manifest,
    directory,
    device,
    session_steps,
    micro_batch=2,
    resume=None,
    smoke=False,
    progress=True,
):
    """One OT plan per data batch; micro-batches only split network backpropagation."""
    if session_steps < 0 or micro_batch < 1:
        raise ValueError("Invalid session step or micro-batch count")
    if session_steps == 0 and resume is None:
        raise ValueError("Inference-only mode requires a V3 checkpoint")
    settings = recipe(smoke)
    seed_everything(settings["seed"])
    model = UNet(settings["widths"]).to(device)
    ema = copy.deepcopy(model).eval().requires_grad_(False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings["lr"], weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    matcher = ExactOptimalTransportConditionalFlowMatcher(sigma=settings["sigma"])
    fingerprint = sha256_file(manifest)
    step, batches_seen, best, skipped = 0, 0, -1.0, 0
    if resume is None and directory.exists() and any(directory.iterdir()):
        raise FileExistsError(
            "Non-empty run directory; specify its latest.pt or use a new directory"
        )
    if resume is not None:
        checkpoint = torch.load(resume, map_location="cpu", weights_only=True)
        if checkpoint.get("recipe") != settings or checkpoint["manifest_sha256"] != fingerprint:
            raise ValueError(
                "Resume requires the same V3 recipe and manifest; V1/V2 are incompatible"
            )
        model.load_state_dict(checkpoint["model"])
        ema.load_state_dict(checkpoint["ema"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scaler.load_state_dict(checkpoint["scaler"])
        step, batches_seen = checkpoint["step"], checkpoint["batches_seen"]
        best, skipped = checkpoint["best_ssim"], checkpoint["amp_skipped"]
        torch.set_rng_state(checkpoint["torch_rng"])
        restore_numpy_rng(checkpoint["numpy_rng"])
        if device.type == "cuda" and checkpoint["cuda_rng"] is not None:
            torch.cuda.set_rng_state_all(checkpoint["cuda_rng"][: torch.cuda.device_count()])
    directory.mkdir(parents=True, exist_ok=True)
    train_data = PairedDataset(
        root, manifest, "train", settings["crop"], True, settings["limit_train"]
    )
    val_data = validation_data(root, manifest, settings)
    if {r["group"] for r in train_data.records} & {r["group"] for r in val_data.records}:
        raise ValueError("Train and validation groups overlap")
    loader = DataLoader(
        train_data,
        batch_sampler=TrainingBatches(
            len(train_data), settings["ot_batch"], settings["seed"], batches_seen
        ),
        num_workers=0,
        generator=torch.Generator().manual_seed(settings["seed"] + 1),
    )
    iterator = iter(loader)
    stop = min(settings["total_steps"], step + session_steps)

    def payload():
        return {
            "format": "otcfm_v3",
            "recipe": settings,
            "manifest_sha256": fingerprint,
            "model": model.state_dict(),
            "ema": ema.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scaler": scaler.state_dict(),
            "step": step,
            "batches_seen": batches_seen,
            "best_ssim": best,
            "amp_skipped": skipped,
            "torch_rng": torch.get_rng_state(),
            "numpy_rng": numpy_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else None,
        }

    write_json(directory / "recipe.json", settings)
    bar = tqdm(total=stop, initial=step, desc="OT-CFM updates", disable=not progress)
    consecutive_skips = 0
    while step < stop:
        model.train()
        optimizer.zero_grad(set_to_none=True)
        warmup = settings["warmup"]
        progress_fraction = (step - warmup) / max(1, settings["total_steps"] - warmup)
        factor = (
            (step + 1) / warmup
            if step < warmup
            else 0.5 * (1 + math.cos(math.pi * min(1, progress_fraction)))
        )
        for group in optimizer.param_groups:
            group["lr"] = settings["lr"] * factor
        batch = next(iterator)
        batches_seen += 1
        time, image, velocity, he = ot_batch(matcher, batch["he"], batch["ihc"])
        loss_sum = 0.0
        for first in range(0, len(image), micro_batch):
            section = slice(first, first + micro_batch)
            x, t, c, target = (a[section].to(device) for a in (image, time, he, velocity))
            with torch.autocast(device.type, dtype=torch.float16, enabled=scaler.is_enabled()):
                prediction = model(x, t, c)
                loss = F.mse_loss(prediction.float(), target.float())
            if not torch.isfinite(loss):
                save_checkpoint(directory / "last-safe.pt", payload())
                raise FloatingPointError("Non-finite OT-CFM loss")
            weight = len(x) / len(image)
            scaler.scale(loss * weight).backward()
            loss_sum += loss.detach().item() * weight
        try:
            updated = optimizer_update(model, optimizer, scaler)
        except FloatingPointError:
            save_checkpoint(directory / "last-safe.pt", payload())
            raise
        if not updated:
            skipped += 1
            consecutive_skips += 1
            print(f"AMP overflow: update skipped; scale={scaler.get_scale()}", flush=True)
            if consecutive_skips >= 16:
                save_checkpoint(directory / "last-safe.pt", payload())
                raise FloatingPointError("16 consecutive AMP overflows")
            continue
        consecutive_skips = 0
        with torch.no_grad():
            for average, current in zip(ema.parameters(), model.parameters(), strict=True):
                average.lerp_(current, 1 - settings["ema"])
            for average, current in zip(ema.buffers(), model.buffers(), strict=True):
                average.copy_(current)
        step += 1
        record = {
            "step": step,
            "otcfm_loss": loss_sum,
            "lr": optimizer.param_groups[0]["lr"],
            "amp_skipped": skipped,
        }
        if step % settings["validate_every"] == 0 or step == stop:
            record.update(
                validate(
                    ema,
                    val_data,
                    device,
                    settings,
                    directory / "validation" / str(step),
                )
            )
            if record["val_ssim"] > best:
                best = record["val_ssim"]
                save_checkpoint(directory / "best.pt", payload())
        if step % 200 == 0 or step == stop:
            save_checkpoint(directory / "latest.pt", payload())
        with (directory / "history.jsonl").open("a") as stream:
            stream.write(json.dumps(record) + "\n")
        bar.update(1)
        bar.set_postfix(loss=f"{loss_sum:.4f}", refresh=False)
        if step % 100 == 0 or step == stop:
            print(json.dumps(record), flush=True)
    bar.close()
    if resume is not None and stop <= checkpoint["step"]:
        return Path(resume)
    return directory / "latest.pt"

In [ ]:
resume = Path(RESUME_FROM) if RESUME_FROM else None
if resume is None and (RUN / "latest.pt").is_file(): resume = RUN / "latest.pt"
if resume is not None:
    previous = torch.load(resume, map_location="cpu", weights_only=True)
    if previous.get("recipe") != SETTINGS or previous.get("manifest_sha256") != sha256_file(MANIFEST):
        raise ValueError("Ожидается V3 checkpoint с тем же recipe и split")
    RUN.mkdir(parents=True, exist_ok=True)
    if resume.parent.resolve() != RUN.resolve():
        for name in ("best.pt", "history.jsonl"):
            source_path, target_path = resume.parent/name, RUN/name
            if source_path.is_file() and not target_path.exists(): shutil.copy2(source_path, target_path)
    print("Resume step:", previous["step"])
    del previous
print("Resume:", resume)
checkpoint = train_session(DATA_ROOT, MANIFEST, RUN, DEVICE, SESSION_STEPS,
                           micro_batch=MICRO_BATCH, resume=resume, smoke=SMOKE_TEST)
print("Latest checkpoint:", checkpoint)

## Validation gallery

Compare raw and EMA outputs using the same inputs and noise. These are 12 validation crops. Release-folder prefixes in IDs do not define the current split; the manifest does.


In [ ]:
history_path = RUN/"history.jsonl"
history = [json.loads(line) for line in history_path.read_text().splitlines()] if history_path.exists() else []
validation_rows = [row for row in history if "val_ssim" in row]
fig, axes = plt.subplots(1,3,figsize=(13,3.5))
axes[0].plot([r["step"] for r in history],[r["otcfm_loss"] for r in history]); axes[0].set(ylabel="OT-CFM loss",xlabel="Update")
axes[1].plot([r["step"] for r in validation_rows],[r["val_ssim"] for r in validation_rows],"o-"); axes[1].set(ylabel="Validation SSIM",xlabel="Update")
axes[2].plot([r["step"] for r in history],[r["lr"] for r in history]); axes[2].set(ylabel="Learning rate",xlabel="Update")
fig.tight_layout(); fig.savefig(RUN/"learning-curves.png",dpi=140); plt.show(); plt.close(fig)

state = torch.load(checkpoint,map_location="cpu",weights_only=True)
validation = validation_data(DATA_ROOT,MANIFEST,SETTINGS)
models = {}
for weights in ("model","ema"):
    model = UNet(SETTINGS["widths"]).to(DEVICE)
    model.load_state_dict(state[weights]); models[weights] = model.eval().requires_grad_(False)
OUTPUT = WORK_ROOT / "otcfm-preview" / f"step-{state['step']}"
OUTPUT.mkdir(parents=True,exist_ok=True)
rows = []
for first in range(0,len(validation),3):
    page = [validation[index] for index in range(first,min(first+3,len(validation)))]
    fig,axes = plt.subplots(len(page),4,figsize=(14,3.5*len(page)),squeeze=False)
    for row,item in enumerate(page):
        c = item["he"].unsqueeze(0).to(DEVICE)
        noise = initial_noise(c,[item["id"]],SETTINGS["seed"])
        panels = [(item["he"],"H&E"),(item["ihc"],"Real IHC")]
        for weights,model in models.items():
            prediction = generate(model,c,noise,SETTINGS["ode_steps"])[0].cpu()
            panels.append((prediction,weights))
            rows.append({"id":item["id"],"weights":weights,**pixel_scores(prediction,item["ihc"])})
            save_image(prediction,OUTPUT/"predictions"/weights/f"{item['id']}.png")
        for ax,(image,title) in zip(axes[row],panels):
            ax.imshow(rgb(image)); ax.set_title(title); ax.axis("off")
        axes[row,0].text(0,-0.05,item["id"],transform=axes[row,0].transAxes,fontsize=8)
    fig.tight_layout(); fig.savefig(OUTPUT/f"gallery-{first//3+1:02d}.png",dpi=140,bbox_inches="tight")
    plt.show(); plt.close(fig)
write_json(OUTPUT/"metrics.json",rows)
display([{ "weights":weights, "ssim":float(np.mean([r["ssim"] for r in rows if r["weights"]==weights])),
           "rgb_mse":float(np.mean([r["rgb_mse"] for r in rows if r["weights"]==weights]))}
         for weights in models])

## Condition-swap diagnostic

Keep noise, checkpoint and solver fixed and change H&E. Check whether tissue geometry follows the new input rather than only changing color.


In [ ]:
item = validation[0]
other = next((validation[i] for i in range(1,len(validation))
              if validation[i]["group"] != item["group"]),validation[1])
c = item["he"].unsqueeze(0).to(DEVICE)
noise = initial_noise(c,[item["id"]],SETTINGS["seed"])
model = models["model"]
prediction,frames = generate(model,c,noise,SETTINGS["ode_steps"],trajectory=True)
fig,axes = plt.subplots(1,len(frames)+2,figsize=(3*(len(frames)+2),3))
panels = [(item["he"],"H&E")] + [(frame[0],f"ODE t={time:.2f}")
          for frame,time in zip(frames,np.linspace(0,1,len(frames)))] + [(item["ihc"],"Real IHC")]
for ax,(image,title) in zip(axes,panels):
    ax.imshow(rgb(image)); ax.set_title(title); ax.axis("off")
fig.tight_layout(); fig.savefig(OUTPUT/"ode-trajectory.png",dpi=140); plt.show(); plt.close(fig)
changed = other["he"].unsqueeze(0).to(DEVICE)
swapped = generate(model,changed,noise,SETTINGS["ode_steps"])
fig,axes = plt.subplots(1,4,figsize=(14,4))
for ax,image,title in zip(axes,(item["he"],other["he"],prediction[0],swapped[0]),
                         ("Original H&E","Different H&E","Output original","Output changed H&E")):
    ax.imshow(rgb(image)); ax.set_title(title); ax.axis("off")
fig.suptitle("Same initial noise / weights / Heun steps; only H&E changes")
fig.tight_layout(); fig.savefig(OUTPUT/"condition-swap.png",dpi=140); plt.show(); plt.close(fig)
write_json(OUTPUT/"condition-swap.json",{
    "original_id":item["id"],"changed_id":other["id"],
    "input_rgb_mae":float(np.abs(rgb(item["he"])-rgb(other["he"])).mean()),
    "output_rgb_mae":float(np.abs(rgb(prediction[0])-rgb(swapped[0])).mean()),
})
del models,model,state
if DEVICE.type == "cuda":torch.cuda.empty_cache()

## Interpretation and export

Minibatch RGB OT does not guarantee correct conditional staining or membrane localization. More ODE steps cannot fix an incorrect learned mapping. Save checkpoints and previews through Kaggle Outputs.

[TorchCFM](https://github.com/atong01/conditional-flow-matching) · [OT-CFM](https://arxiv.org/abs/2302.00482).


In [ ]:
archive_path = WORK_ROOT/f"her2-otcfm-v3-{OUTPUT.name}.zip"
with zipfile.ZipFile(archive_path,"w",zipfile.ZIP_DEFLATED) as archive:
    for directory in (RUN,OUTPUT,METHOD_DIR):
        for path in sorted(directory.rglob("*")):
            if path.is_file():archive.write(path,path.relative_to(WORK_ROOT))
print("Checkpoints:",RUN,"Preview:",OUTPUT)
display(FileLink(str(archive_path)))